# Interactive Figure 15 — Formation Efficiency by Channel

Interactive Plotly version of `Fig15_combined_with_and_without_CE_4x6.pdf`.

**Layout:** 4 rows × 6 columns
- Rows: Iorio et al. | Broekgaarden et al. | van Son et al. | Neijssel et al.
- Cols 1–3: fraction **with CE** — BBH, BHNS, BNS
- Cols 4–6: fraction **without CE** — BBH, BHNS, BNS

**Hover:** move your mouse over any line to highlight it and see its model label.

**Output:** `interactive_figures_and_tables/Fig15_combined_with_and_without_CE_interactive.html`

In [ ]:
import os
import pickle
import string
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
path_github = "/Users/floorbroekgaarden/Projects/GitHub/Rates_of_Formation_Channels/"
data_dir    = path_github + "fc_data/data_for_formation_efficiency/"
iorio_loc   = data_dir + "Iorio/"
broek_dir   = data_dir + "Broekgaarden22/"
nei_csv     = data_dir + "processed_channels.csv"
vson_pkl    = data_dir + "vanSon23_yield_data.pkl"
path_save   = path_github + "interactive_figures_and_tables/"

# ── Colors (match the original matplotlib figure) ────────────────────────────
COLOR_CE    = "#00A7E1"   # blue  — with CE
COLOR_NOCE  = "#FFA630"   # orange — without CE

# ── Figure geometry ──────────────────────────────────────────────────────────
NROWS, NCOLS = 4, 6
ROW_LABELS   = ["Iorio et al.", "Broekgaarden et al.", "van Son et al.", "Neijssel et al."]

# (row_idx 0-based, col_idx 0-based) → (study, dco, fraction)
# Cols 0-2 = with_CE for BBH/BHNS/BNS; cols 3-5 = without_CE for BBH/BHNS/BNS
DCO_ORDER    = ["BBH", "BHNS", "BNS"]
FRAC_ORDER   = ["with_CE", "with_CE", "with_CE",
                "without_CE", "without_CE", "without_CE"]
DCO_PER_COL  = ["BBH", "BHNS", "BNS", "BBH", "BHNS", "BNS"]

In [ ]:
# ── Iorio data ───────────────────────────────────────────────────────────────

iorio_model_dic = {
    "F":     os.path.join(iorio_loc, "simulation_F"),
    "QCBSE": os.path.join(iorio_loc, "simulation_QCBSE"),
    "QCBB":  os.path.join(iorio_loc, "simulation_QCBB"),
    "RBSE":  os.path.join(iorio_loc, "simulation_RBSE"),
    "NTC":   os.path.join(iorio_loc, "simulation_NTC"),
    "NT":    os.path.join(iorio_loc, "simulation_NT"),
    "QHE":   os.path.join(iorio_loc, "simulation_QHE"),
    "F19":   os.path.join(iorio_loc, "simulation_F19"),
    "K150":  os.path.join(iorio_loc, "simulation_K150"),
    "K265":  os.path.join(iorio_loc, "simulation_K265"),
    "LX":    os.path.join(iorio_loc, "simulation_LX"),
    "LK":    os.path.join(iorio_loc, "simulation_LK"),
    "LC":    os.path.join(iorio_loc, "simulation_LC"),
    "SND":   os.path.join(iorio_loc, "simulation_SND"),
    "OPT":   os.path.join(iorio_loc, "simulation_OPT"),
    "F5M":   os.path.join(iorio_loc, "simulation_F5M"),
}

# bco_type key in summary.csv → DCO label used in the figure
IORIO_BCO = {"BBH": "BHBHm", "BHNS": "BHNSm", "BNS": "NSNSm"}


def iorio_fractions(model, path, dco):
    """Return list of (label, logZ, frac_with_CE, frac_without_CE) for every alpha."""
    csv = os.path.join(path, "summary.csv")
    if not os.path.exists(csv):
        return []
    df  = pd.read_csv(csv)
    bco = IORIO_BCO[dco]
    results = []
    for alpha in sorted(df["alpha"].dropna().unique()):
        sub = df[df["alpha"] == alpha].sort_values("Z")
        z   = sub["Z"].to_numpy(float)
        tot = sub[bco].to_numpy(float)
        ci  = sub[f"{bco}I"].to_numpy(float)
        cii = sub[f"{bco}II"].to_numpy(float)
        ciii= sub[f"{bco}III"].to_numpy(float)
        civ = sub[f"{bco}IV"].to_numpy(float)
        with np.errstate(divide="ignore", invalid="ignore"):
            f_w   = np.where(tot > 0, (ci + ciii + civ) / tot, np.nan)
            f_now = np.where(tot > 0, cii / tot, np.nan)
        # mask BNS rows with zero CE count
        if dco == "BNS":
            bad = (ci + ciii + civ) <= 0
            f_w[bad] = np.nan
        logZ = np.log10(z)
        label = f"Iorio | model {model} | α={alpha}"
        results.append((label, logZ, f_w, f_now))
    return results


# Pre-load all Iorio data
iorio_data = {dco: [] for dco in DCO_ORDER}
for model, path in iorio_model_dic.items():
    for dco in DCO_ORDER:
        iorio_data[dco].extend(iorio_fractions(model, path, dco))

print(f"Iorio traces loaded: BBH={len(iorio_data['BBH'])}, BHNS={len(iorio_data['BHNS'])}, BNS={len(iorio_data['BNS'])}")

In [ ]:
# ── Broekgaarden data ────────────────────────────────────────────────────────

BROEK_DCO_FILE = {"BBH": "BHBH", "BHNS": "BHNS", "BNS": "NSNS"}


def broek_fractions(dco):
    """Return list of (label, logZ, frac_with_CE, frac_without_CE) for every model letter."""
    fname = f"formation_channels_per_Z_hdf5weights_{BROEK_DCO_FILE[dco]}.csv"
    fpath = os.path.join(broek_dir, fname)
    if not os.path.exists(fpath):
        print(f"Missing: {fpath}")
        return []
    df    = pd.read_csv(fpath, index_col=0)
    zvals = np.array([float(zi.replace("_", ".")) for zi in df.index])
    logZ  = np.log10(zvals)

    results = []
    # All model letters present in the columns
    letters = sorted({c.split(" ")[0] for c in df.columns})
    for m in letters:
        try:
            ch1   = df[f"{m} channel I  [Msun^{{{{-1}}}}]"].to_numpy(float)
            ch2   = df[f"{m} channel II  [Msun^{{{{-1}}}}]"].to_numpy(float)
            ch3   = df[f"{m} channel III  [Msun^{{{{-1}}}}]"].to_numpy(float)
            ch4   = df[f"{m} channel IV  [Msun^{{{{-1}}}}]"].to_numpy(float)
            total = df[f"{m} All  [Msun^{{{{-1}}}}]"].to_numpy(float)
        except KeyError:
            continue
        with np.errstate(divide="ignore", invalid="ignore"):
            f_w   = np.where(total > 0, (ch1 + ch3 + ch4) / total, np.nan)
            f_now = np.where(total > 0,  ch2 / total,               np.nan)
        results.append((f"Broekgaarden | model {m}", logZ, f_w, f_now))
    return results


broek_data = {dco: broek_fractions(dco) for dco in DCO_ORDER}
print(f"Broekgaarden traces: BBH={len(broek_data['BBH'])}, BHNS={len(broek_data['BHNS'])}, BNS={len(broek_data['BNS'])}")

In [ ]:
# ── van Son data ─────────────────────────────────────────────────────────────

sim_name_dict = {
    "NewWinds_RemFryer2012":             "Fiducial",
    "NewWinds_RemMullerMandel":          "MM remnant",
    "NewWinds_RemFryer2012_noNSBHkick":  "No natal kicks",
    "NewWinds_RemFryer2012_noWRwinds":   "No WR winds",
    "NewWinds_RemFryer2012_noMSwinds":   "No MS winds",
    "NewWinds_RemFryer2012_oldRSG":      "Old RSG",
    "OldWinds_RemFryer2012":             "Old Winds",
}

VSON_DCO = {"BBH": "BBH", "BHNS": "BHNS", "BNS": "NSNS"}

with open(vson_pkl, "rb") as f:
    yield_data = pickle.load(f)


def vson_fractions(dco):
    """Return list of (label, logZ, frac_with_CE, frac_without_CE)."""
    flavor  = VSON_DCO[dco]
    results = []
    for sim, label in sim_name_dict.items():
        if sim not in yield_data:
            continue
        data  = yield_data[sim]
        zvals = np.log10(np.array(data["Z_values"], dtype=float))
        fd    = data.get(flavor, {})
        ce    = np.array(fd.get("CE_group_lengths",     []), dtype=float)
        stab  = np.array(fd.get("stable_group_lengths", []), dtype=float)
        che   = np.array(fd.get("CHEstable_group_lengths", np.zeros_like(ce)), dtype=float) if dco == "BBH" else np.zeros_like(ce)
        denom = ce + stab + che
        with np.errstate(divide="ignore", invalid="ignore"):
            f_w   = np.where(denom > 0, ce / denom,          np.nan)
            f_now = np.where(denom > 0, (stab + che) / denom, np.nan)
        results.append((f"van Son | {label}", zvals, f_w, f_now))
    return results


vson_data = {dco: vson_fractions(dco) for dco in DCO_ORDER}
print(f"van Son traces: BBH={len(vson_data['BBH'])}, BHNS={len(vson_data['BHNS'])}, BNS={len(vson_data['BNS'])}")

In [ ]:
# ── Neijssel data (BBH only) ─────────────────────────────────────────────────

def nei_fractions():
    """Return (label, logZ, frac_with_CE, frac_without_CE) for Neijssel BBH."""
    df    = pd.read_csv(nei_csv)
    # log10Z_Zsun = log10(Z/Z_sun); convert to log10(Z) with Z_sun = 0.014
    logZ  = df["log10Z_Zsun"].to_numpy(float) + np.log10(0.014)
    f_w   = (df["f_I"] + df["f_III"]).to_numpy(float)  # ch I (classic CE) + ch III (immediate CE)
    f_now = df["f_II"].to_numpy(float)                  # ch II (stable MT)
    return [("Neijssel et al. (2019) | BBH", logZ, f_w, f_now)]


nei_data = nei_fractions()
print(f"Neijssel traces: {len(nei_data)}")

In [ ]:
# ── Other individual study points (row 4 scatter) ────────────────────────────
# Zsun = 0.014. logZ values outside [-4.1, -1.45] are clipped to the left edge.

Zsun  = 0.014
X_MIN = -4.1   # left edge of x-axis

OTHER_STUDY_POINTS = [
    # BNS
    dict(dco="BNS",  study="Mestichelli et al. (2025)", logZ=X_MIN,              frac=0.919, channel="with_CE",    relation="lower"),
    dict(dco="BNS",  study="Mestichelli et al. (2025)", logZ=-4,                 frac=0.993, channel="with_CE",    relation="lower"),
    dict(dco="BNS",  study="Dominik et al. (2012)",     logZ=np.log10(0.1*Zsun), frac=0.918, channel="with_CE",    relation="lower"),
    dict(dco="BNS",  study="Andrews et al. (2015)",     logZ=np.log10(Zsun),     frac=1.00,  channel="with_CE",    relation="approx"),
    dict(dco="BNS",  study="Belczynski et al. (2002)",  logZ=np.log10(0.02),     frac=0.95,  channel="with_CE",    relation="lower"),
    dict(dco="BNS",  study="Chruslinska et al. (2018)", logZ=np.log10(Zsun),     frac=0.95,  channel="with_CE",    relation="lower"),
    dict(dco="BNS",  study="Chattaraj et al. (2026)",   logZ=np.log10(Zsun),     frac=0.95,  channel="with_CE",    relation="lower"),
    dict(dco="BNS",  study="Chattaraj et al. (2026)",   logZ=np.log10(Zsun),     frac=0.06,  channel="without_CE", relation="lesssim"),
    dict(dco="BNS",  study="Dominik et al. (2012)",     logZ=np.log10(Zsun),     frac=0.942, channel="with_CE",    relation="lower"),
    dict(dco="BNS",  study="Vigna-Gomez et al. (2018)", logZ=np.log10(Zsun),     frac=0.91,  channel="with_CE",    relation="lower"),
    # BHNS
    dict(dco="BHNS", study="Mestichelli et al. (2025)", logZ=X_MIN,              frac=0.36,  channel="with_CE",    relation="lower"),
    dict(dco="BHNS", study="Mestichelli et al. (2025)", logZ=X_MIN,              frac=0.559, channel="without_CE", relation="lower"),
    dict(dco="BHNS", study="Mestichelli et al. (2025)", logZ=-4,                 frac=0.462, channel="with_CE",    relation="lower"),
    dict(dco="BHNS", study="Mestichelli et al. (2025)", logZ=-4,                 frac=0.469, channel="without_CE", relation="lower"),
    dict(dco="BHNS", study="Dominik et al. (2012)",     logZ=np.log10(0.1*Zsun), frac=0.936, channel="with_CE",    relation="lower"),
    dict(dco="BHNS", study="Belczynski et al. (2002)",  logZ=np.log10(0.02),     frac=0.80,  channel="with_CE",    relation="lower"),
    dict(dco="BHNS", study="Dominik et al. (2012)",     logZ=np.log10(Zsun),     frac=0.972, channel="with_CE",    relation="lower"),
    dict(dco="BHNS", study="Xing et al. (2024b)",       logZ=np.log10(Zsun),     frac=0.70,  channel="with_CE",    relation="approx"),
    dict(dco="BHNS", study="Xing et al. (2024b)",       logZ=np.log10(Zsun),     frac=0.30,  channel="without_CE", relation="approx"),
    # BBH
    dict(dco="BBH",  study="Belczynski et al. (2002)",  logZ=np.log10(0.02),     frac=0.95,  channel="with_CE",    relation="lower"),
    dict(dco="BBH",  study="Belczynski et al. (2022)",  logZ=np.log10(0.4*Zsun), frac=0.75,  channel="with_CE",    relation="approx"),
    dict(dco="BBH",  study="Briel et al. (2026)",       logZ=np.log10(0.2*Zsun), frac=1.0,   channel="with_CE",    relation="approx"),
    dict(dco="BBH",  study="Dominik et al. (2012)",     logZ=np.log10(Zsun),     frac=0.989, channel="with_CE",    relation="lower"),
    dict(dco="BBH",  study="Dominik et al. (2012)",     logZ=np.log10(0.1*Zsun), frac=0.96,  channel="with_CE",    relation="lower"),
]

# Distinct colour per study
ALL_STUDIES  = sorted({p["study"] for p in OTHER_STUDY_POINTS})
_PALETTE     = ["#e41a1c","#377eb8","#4daf4a","#984ea3","#ff7f00",
                "#a65628","#f781bf","#999999","#66c2a5","#fc8d62"]
STUDY_COLORS = {s: _PALETTE[i % len(_PALETTE)] for i, s in enumerate(ALL_STUDIES)}

# Marker symbol: circle = approximate value; triangle-up = lower bound (≳); triangle-down = upper bound (≲)
REL_SYMBOL = {"approx": "circle", "lower": "triangle-up", "lesssim": "triangle-down"}

print(f"Other-study points: {len(OTHER_STUDY_POINTS)} across {len(ALL_STUDIES)} studies")

In [ ]:
# ── Build Plotly figure ───────────────────────────────────────────────────────

col_titles = [
    "BBH — with CE", "BHNS — with CE", "BNS — with CE",
    "BBH — without CE", "BHNS — without CE", "BNS — without CE",
]

fig = make_subplots(
    rows=NROWS, cols=NCOLS,
    shared_xaxes=True, shared_yaxes=True,
    subplot_titles=col_titles,
    horizontal_spacing=0.02, vertical_spacing=0.06,
)

BASE_OPACITY = 0.65

# ── Rows 1–3: line traces (Iorio, Broekgaarden, van Son) ─────────────────────
for row_0, study_dict in enumerate([iorio_data, broek_data, vson_data]):
    row_1 = row_0 + 1
    for col_0 in range(NCOLS):
        dco = DCO_PER_COL[col_0]; frac_key = FRAC_ORDER[col_0]
        color = COLOR_CE if frac_key == "with_CE" else COLOR_NOCE
        for (label, logZ, f_w, f_now) in study_dict.get(dco, []):
            y = f_w if frac_key == "with_CE" else f_now
            fig.add_trace(go.Scatter(
                x=logZ.tolist(), y=y.tolist(), mode="lines", name=label,
                line=dict(color=color, width=1.2), opacity=BASE_OPACITY, showlegend=False,
                hovertemplate=f"<b>{label}</b><br>log₁₀(Z): %{{x:.2f}}<br>Fraction: %{{y:.3f}}<extra></extra>",
            ), row=row_1, col=col_0 + 1)

# ── Row 4: Neijssel line (BBH panels only) ────────────────────────────────────
for col_0 in range(NCOLS):
    dco = DCO_PER_COL[col_0]; frac_key = FRAC_ORDER[col_0]
    color = COLOR_CE if frac_key == "with_CE" else COLOR_NOCE
    if dco == "BBH":
        for (label, logZ, f_w, f_now) in nei_data:
            y = f_w if frac_key == "with_CE" else f_now
            fig.add_trace(go.Scatter(
                x=logZ.tolist(), y=y.tolist(), mode="lines", name=label,
                line=dict(color=color, width=1.8), opacity=BASE_OPACITY, showlegend=False,
                hovertemplate=f"<b>{label}</b><br>log₁₀(Z): %{{x:.2f}}<br>Fraction: %{{y:.3f}}<extra></extra>",
            ), row=4, col=col_0 + 1)

# ── Row 4: scatter points from OTHER_STUDY_POINTS ────────────────────────────
for col_0 in range(NCOLS):
    dco = DCO_PER_COL[col_0]; frac_key = FRAC_ORDER[col_0]
    pts = [p for p in OTHER_STUDY_POINTS if p["dco"] == dco and p["channel"] == frac_key]
    for p in pts:
        study = p["study"]; rel = p["relation"]
        tip = (
            f"<b>{study}</b><br>"
            f"log₁₀(Z): {p['logZ']:.2f}<br>"
            f"Fraction: {p['frac']:.3f}<br>"
            f"({rel})<extra></extra>"
        )
        fig.add_trace(go.Scatter(
            x=[p["logZ"]], y=[p["frac"]],
            mode="markers", name=study,
            marker=dict(symbol=REL_SYMBOL.get(rel, "circle"), size=9,
                        color=STUDY_COLORS[study], line=dict(width=1, color="black")),
            opacity=BASE_OPACITY, showlegend=False,
            hovertemplate=tip,
        ), row=4, col=col_0 + 1)

print(f"Total traces: {len(fig.data)}")

In [ ]:
# ── Layout & annotations ─────────────────────────────────────────────────────

fig.update_layout(
    height=820,
    width=1500,
    plot_bgcolor="white",
    paper_bgcolor="white",
    hovermode="closest",
    hoverlabel=dict(bgcolor="white", font_size=12, namelength=-1),
    margin=dict(l=70, r=20, t=100, b=60),
    showlegend=False,
    title=dict(
        text="Formation efficiency by channel and study — hover to identify a line",
        x=0.5, font=dict(size=14),
    ),
)

# Shared axis ranges
fig.update_xaxes(range=[-4.1, -1.45], showgrid=True, gridcolor="rgba(0,0,0,0.10)",
                 dtick=0.5, tickfont=dict(size=9))
fig.update_yaxes(range=[-0.05, 1.05], showgrid=True, gridcolor="rgba(0,0,0,0.10)",
                 dtick=0.25, tickfont=dict(size=9))

# x-axis label on bottom row only
for col_1 in range(1, NCOLS + 1):
    fig.update_xaxes(title_text="log₁₀(Z)", title_font=dict(size=10), row=NROWS, col=col_1)

# y-axis label on leftmost column only
for row_1 in range(1, NROWS + 1):
    fig.update_yaxes(title_text="fraction", title_font=dict(size=10), row=row_1, col=1)

# Row labels (right side)
row_y_paper = [0.875, 0.635, 0.395, 0.155]   # approximate paper y centres
for i, (rl, yp) in enumerate(zip(ROW_LABELS, row_y_paper)):
    fig.add_annotation(
        x=1.005, y=yp,
        xref="paper", yref="paper",
        text=f"<b>{rl}</b>",
        showarrow=False,
        xanchor="left", yanchor="middle",
        font=dict(size=10, color="#333"),
        textangle=-90,
    )

# Dividing line between with-CE and without-CE column groups
fig.add_shape(
    type="line",
    x0=0.502, x1=0.502, y0=0, y1=1,
    xref="paper", yref="paper",
    line=dict(color="#888", width=1.5, dash="dot"),
)

print("Layout configured.")

In [ ]:
# ── Hover-highlight JavaScript ────────────────────────────────────────────────
# Lines → bold & fully opaque on hover, all others fade.
# Scatter points → fully opaque on hover, all others fade.
# On mouse-out everything restores.

HIGHLIGHT_JS = """
(function() {
    var gd = document.getElementsByClassName('plotly-graph-div')[0];
    var baseOpacity = 0.65, hlOpacity = 1.0, dimOpacity = 0.10;
    var baseWidth   = 1.2,  hlWidth   = 3.5;

    gd.on('plotly_hover', function(data) {
        var pn = data.points[0].curveNumber;
        var n  = gd.data.length;
        var opacs = [], widths = [];
        for (var i = 0; i < n; i++) {
            opacs.push(i === pn ? hlOpacity : dimOpacity);
            var isLine = gd.data[i].mode && gd.data[i].mode.indexOf('lines') !== -1;
            widths.push(isLine ? (i === pn ? hlWidth : baseWidth) : 1);
        }
        Plotly.restyle(gd, {'opacity': opacs, 'line.width': widths});
    });

    gd.on('plotly_unhover', function() {
        var n = gd.data.length, opacs = [], widths = [];
        for (var i = 0; i < n; i++) {
            opacs.push(baseOpacity);
            var isLine = gd.data[i].mode && gd.data[i].mode.indexOf('lines') !== -1;
            widths.push(isLine ? baseWidth : 1);
        }
        Plotly.restyle(gd, {'opacity': opacs, 'line.width': widths});
    });
})();
"""

In [ ]:
# ── Save & show ───────────────────────────────────────────────────────────────

out_path = path_save + "Fig15_combined_with_and_without_CE_interactive.html"

fig.write_html(
    out_path,
    post_script=HIGHLIGHT_JS,
    include_plotlyjs="cdn",   # load Plotly from CDN → smaller file
)
print(f"Saved: {out_path}")

fig.show()